## **SAVE A COPY OF THIS NOTEBOOK TO PUT ANSWERS INTO**

#Part 1 (Reading Assignment), 20 points

**Reminder: there are 5 reading assignments, 3% each for 15% total of your final grade.**

The reading assignment consists of two papers - [one on alignment before fusion](https://arxiv.org/abs/2107.07651), and one on the [Platonic Representation Hypothesis](https://arxiv.org/pdf/2405.07987). Read both papers and then answer the following questions:

1. Explain the implications of 'align before fuse' on your tasks of interest. at which degree, and what type of fusion needs to be performed? And what level of alignment would you need to perform for your data, so that subsequent fusion or representation learning is successful?

For our problem, we need to make sure the video and labels are aligned on the task that is performed (task labels), and potentially requires the attention masks be aligned with the prediction labels on different time scales. It will also be good for the audio be aligned with the tasks too.

2. How can you perform controlled experiments to validate the types of fusion and/or alignment needed for your tasks? What are some challenges you foresee in fusing and aligning your data?

Using the vision and audio tokens to predict tasks labels will be an easy way to check if the alignment is done. For fusion, we can do ablation studies on prediction of task labels by disregarding audio or image. For us, the domain of vision and tasks labels are pretty heterogenous, which will be the main challenge of alignment.

3. Explain the implications of 'platonic representation hypothesis' on your tasks of interest. Do you believe alignment between modalities would automatically emerge as models trained on your data are scaled up?

The platonic represntation hypothesis implies that if we have enough data and pretrain enough, we can use less modalities to predict the features we want. Personally i dont think alignemt will automatically show up... but it will be nice if it does.

4. What are some reasons why alignment would not emerge i.e., counter-arguments to the platonic representation hypothesis? You are encouraged to search for follow-up works to the original paper that both support and counter the original arguments.

One simple reason is that one modality might not completly describe the information needed. Another good reason is that the metrics we use to judge alignment might be too low dimensional and we aren't capturing the full space of representations.

5. What experiments would you propose to validate the existence and emergence of alignment in your tasks?

I'd check the embedding space and see if there are learned similarities, or simply to predict tasks/check the distance between paired concepts and see if there are similarities.

6. Can you also think of some downsides of strongly (or perfectly) aligning your data modalities? How can you design experiments to validate that these risks are not present in your trained models?

When they are suoer aligneed, we lose heterogenous parts of the modalities that can provide more infromations. I'd check the ranked space of the encoded tokens, and see how much information is contained before/after the encodings. One can also do abalation studies, when the modalities are perfectly aligned losing some modalitles shouldn't impact the outcomes.

#Part 2 (Homework Assignment), 100 points

**Reminder: there are 5 homework assignments, 7% each for 35% total of your final grade.**

For this assignment, we will finally begin playing with some of the concepts discussed in the class regarding multimodal modeling.

The first part will deal with Einsum and Tensors.

# Problem 1: Tensors (5 points)

(5 pts) Let's start with tensors. A tensor represents an N-th dimensional array of numbers. In machine learning, they are used to represent data as they can efficiently represent complex data to train with. We traditionally use PyTorch as the package of choice to work with tensors. Fill in the code below with the right tensor operations. Feel free to consult the documentation and the PyTorch tutorials for help.

In [2]:
import torch
mat_A = torch.rand(3, 2)
mat_B = torch.rand(2, 3)

In [5]:
# Common PyTorch operations

# Adding
mat_C = mat_A + mat_B.T

# Transpose
mat_A_transpose = mat_A.T

# Matrix multiplication
mat_mult = torch.matmul(mat_A, mat_B)

# Element-wise multiplication
mat_mult_elm = mat_A * mat_B.T

# Create a tensor of size (4, 4) of ones
ones = torch.ones(4, 4)

# Compute mean of A
mean_A = torch.mean(mat_A)

# Problem 2: Einsum (5 points)

(10 pts)
Now lets proceed with Einsum. This is a powerful, compact notation used for expressing complex tensor operations on multi-dimensional arrays using a simple string of index labels.

Here is a quick example of using einsum to multiply two matrices.

In [6]:
A = torch.rand(3, 2)
B = torch.rand(2, 3)

C = torch.einsum('ij,jk->ik', A, B)
print(C)

tensor([[0.0217, 0.0181, 0.0711],
        [0.1500, 0.2035, 0.5663],
        [0.0808, 0.0738, 0.2710]])


The labels provide a shorthand as to what operation to do. Think of the left index as what is before, and the right as to what the dimensions of the final product should look like.

Now use this to do the other possible operations:

In [10]:
a = torch.rand(3, 1)
b = torch.rand(3, 1)

A = torch.rand(3, 2)
B = torch.rand(2, 3)

# Dot Product of a and b
d_prod = torch.einsum('ij,ij->', a, b)

# Transpose using vector b
transpose = torch.einsum('ij->ji', b)

# Summation (element-wise and column-wise of A)
sum_element = torch.einsum('ij->', A)
sum_column = torch.einsum('ij->j', A)

# Diagonal of A
diag = torch.einsum('ii->i', A[:2, :2])

# Outer Product of A and B
outer = torch.einsum('ij,kl->ijkl', A, B)

In [11]:
# Tests to verify that operations were done correctly
def to_list(t):
    return t.detach().cpu().tolist()

def check_dot_product(ans, a, b):
    expected = sum(i[0] * j[0] for i, j in zip(to_list(a), to_list(b)))
    assert torch.allclose(ans, torch.tensor(expected))

def check_transpose(ans, b):
    b_list = to_list(b)
    expected = [[row[i] for row in b_list] for i in range(len(b_list[0]))]
    assert torch.allclose(ans, torch.tensor(expected))

def check_sum_element(ans, A):
    expected = sum(val for row in to_list(A) for val in row)
    assert torch.allclose(ans, torch.tensor(expected))

def check_sum_column(ans, A):
    A_list = to_list(A)
    expected = [sum(row[i] for row in A_list) for i in range(len(A_list[0]))]
    assert torch.allclose(ans, torch.tensor(expected))

def check_diagonal(ans, A):
    A_list = to_list(A)
    expected = [A_list[i][i] for i in range(min(len(A_list), len(A_list[0])))]
    assert torch.allclose(ans, torch.tensor(expected))

def check_outer_product(ans, a, b):
    a_l, b_l = to_list(a.squeeze(1)), to_list(b.squeeze(1))
    expected = [[i * j for j in b_l] for i in a_l]
    assert torch.allclose(ans, torch.tensor(expected))

# Problem 3: Unimodal Models (10 points)

We now explore unimodal models and multimodal fusion. For the first part we will work on the image and audio digit dataset AV-MNIST to do digit classification. To benchmark effectiveness, we will use the [Multibench](https://arxiv.org/abs/2107.07502) benchmark. First, we will clone the repo, and get the necessary packages and dataset.

**Note: MAKE SURE YOU SWITCH TO A GPU TO RUN THE MODELS. RUNTIME -> CHANGE RUNTIME TYPE -> T4 GPU (or any other). Be mindful of Google's GPU limits based on what kind of account you own.**

**Also, if you are a student you should be able to have Colab Pro for free if you don't already. Take advantage of that!**

**THIS IS AN EXAMPLE, DO NOT BE RESTRICTED BY WHAT WE DO HERE WHEN YOU HAVE TO IMPLEMENT THIS FOR YOUR OWN DATASET.**

# Getting repo

In [1]:
!git clone https://github.com/pliang279/MultiBench.git
%cd MultiBench

fatal: destination path 'MultiBench' already exists and is not an empty directory.
/content/MultiBench


# Getting AV-MNIST dataset

In [2]:
!mkdir data
!pip install gdown
!pip install torch==2.3.1 torchvision==0.18.1 torchtext==0.18.0 torchaudio==2.3.1
!pip install memory_profiler

mkdir: cannot create directory ‘data’: File exists


In [ ]:
!gdown 1KvKynJJca5tDtI5Mmp6CoRh9pQywH8Xp
!tar -xvzf avmnist.tar.gz

In [ ]:
from google.colab import files
uploaded = files.upload()

In [4]:
# 1. Path to the folder you untarred
data_dir = '/content/MultiBench/avmnist'

from datasets.avmnist.get_data import get_dataloader
traindata, validdata, testdata  = get_dataloader(data_dir, batch_size=256)

FileNotFoundError: [Errno 2] No such file or directory: '/content/MultiBench/avmnist/image/train_data.npy'

# Getting packages

In [ ]:
import torch
import torch.nn as nn
import sys
import os
import torch.optim as optim
from tqdm import tqdm
from unimodals.common_models import GRU, MLP, Sequential, Identity
from training_structures.Supervised_Learning import train, test

We will now start by creating, training, and testing unimodal models for each of the AV-MNIST modalities.

# Audio

In [ ]:
class AudioModel(nn.Module):
    def __init__(self, input_dim=12544, hidden_dim=64, dropout_probability=0.2):
        super(AudioModel, self).__init__()
        self.conv = nn.Sequential(
            # Start with a stride of 2 to instantly cut data in half
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), # 112 -> 56
            nn.ReLU(),
            nn.MaxPool2d(2),                                     # 56 -> 28
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 28 -> 14
            nn.ReLU(),
            nn.Flatten() # Only 6272 features now!
        )
        self.fc = nn.Linear(6272, 10)

    def forward(self, x):
        x = x.view(-1, 1, 112, 112)
        return self.fc(self.conv(x))

# Image

In [ ]:
import torch.nn.functional as F

class ImageModel(nn.Module):
    def __init__(self, dropout_prob=0.2):
        super(ImageModel, self).__init__()

        # input: [batch, 1, 28, 28]
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # Reduces size by half

        # After two poolings: 28 -> 14 -> 7
        # Final flattened size: 64 channels * 7 * 7
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten all dimensions except batch
        x = torch.flatten(x, 1)
        return self.fc(x)

# Training and Testing

We use cross-entropy due to this being a classification task

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler

# We use a scalar here to reduce system RAM use (to avoid crashing the session) while not impacting performance.
scaler = GradScaler()

def train_and_test_unimodal(model, train_loader, valid_loader, test_loader, modality_idx, epochs=5, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
    model.to(device)

    # Use CrossEntropyLoss for a classification task
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)

    best_valid_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            # batch[0] = images, batch[1] = audio
            x = batch[modality_idx].to(device).float()

            # Classification labels must be Long tensors, not Float
            y = batch[2].to(device).long().squeeze()

            optimizer.zero_grad()

            with autocast(device_type=device.type, enabled=(device.type != 'cpu')):
                outputs = model(x)
                loss = criterion(outputs, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        # --- Validation Phase ---
        model.eval()
        valid_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in valid_loader:
                x = batch[modality_idx].to(device).float()
                y = batch[2].to(device).long().squeeze()

                outputs = model(x)
                valid_loss += criterion(outputs, y).item()

                _, predicted = torch.max(outputs.data, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()

        avg_train = train_loss / len(train_loader)
        avg_valid = valid_loss / len(valid_loader)
        accuracy = 100 * correct / total

        if avg_valid < best_valid_loss:
            best_valid_loss = avg_valid
            torch.save(model.state_dict(), 'best_avmnist_model.pt')

        print(f"Epoch {epoch}: Train Loss: {avg_train:.4f} | Valid Acc: {accuracy:.2f}%")

    # Final Testing follows the same logic (CrossEntropy + Index 2)
    print("\n--- Final Evaluation Complete ---")
    model.load_state_dict(torch.load('best_avmnist_model.pt'))
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            x = batch[modality_idx].to(device).float()
            y = batch[2].to(device).long().squeeze()

            outputs = model(x)
            test_loss += criterion(outputs, y).item()

            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

        test_accuracy = 100 * correct / total
        test_loss /= len(test_loader)
        print(f"Final Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.2f}%")

# Training and testing for each modality:

# Audio:

In [ ]:
audio_model = AudioModel()
train_and_test_unimodal(audio_model, traindata, validdata, testdata, modality_idx=1)

Epoch 0: Train MAE: 1.3108 | Valid MAE: 1.3898
Epoch 10: Train MAE: 1.2635 | Valid MAE: 1.3722
Epoch 20: Train MAE: 1.1997 | Valid MAE: 1.3994
Epoch 30: Train MAE: 1.0816 | Valid MAE: 1.4532
Epoch 40: Train MAE: 0.8958 | Valid MAE: 1.4853

--- Final Test Evaluation ---
Final Test MAE: 1.4729


# Image:

In [ ]:
image_model = ImageModel()
train_and_test_unimodal(image_model, traindata, validdata, testdata, modality_idx=0)

Answer the following questions:

1. (5 points) Try to get the best performance out of each model by playing around with hyperparameters (hint: you may have to playing around and even add additional arguments to the layers like dropout, look at the documentation and look into how we can improve performace). List the best performance you were able to get and the hyperparameters you used.
2. (5 points) Compare the performances of each modality. What do these suggest to you? What could be done to get the worst performing ones to get closer to the best performing modality/model?

# Problem 4: Multimodal Fusion (10 points)

Now you will play with multimodal fusion. Lets use a late fusion to improve our performance. We have provided some code with the hyperparameters to consider, but you are encouraged to play with them to try to get improvments. To make things simpler, the encoders for both modalities have been provided. However, some other parts are missing, so you will have to fill those in!

In [ ]:
import torch.nn as nn
from unimodals.common_models import MLP
from fusions.common_fusions import MultiplicativeInteractions2Modal, Concat
from training_structures.Supervised_Learning import train

# Image Encoder
class ImageEncoder(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, output_dim)
        )
    def forward(self, x):
        return self.conv(x)

# Audio Encoder
class AudioEncoder(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(), # 112 -> 56
            nn.MaxPool2d(2), # 56 -> 28
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(), # 28 -> 14
            nn.Flatten(),
            nn.Linear(64 * 14 * 14, output_dim)
        )
    def forward(self, x):
        return self.conv(x)

encoders = [ImageEncoder(output_dim=64).cuda(), AudioEncoder(output_dim=64).cuda()]

# Concat fusion
fusion = Concat()

# Head MLP: input 128 -> hidden 256 -> output 10
head = MLP(128, 256, 10).cuda()

# Run Training
print("Starting Training...")
train(encoders, fusion, head, traindata, validdata, 5,
      task="classification", optimtype=torch.optim.AdamW, is_packed=False,
      lr=5e-4, save='avmnist_lmf.pt', weight_decay=0.001,
      objective=torch.nn.CrossEntropyLoss())

# Run Test
model = torch.load('avmnist_lmf.pt').cuda()
test(model, testdata, 'avmnist', is_packed=False, task="classification",
      criterion=torch.nn.CrossEntropyLoss(), no_robust=True)

Training with Low Rank Tensor Fusion (LMF):


/content/MultiBench/fusions/common_fusions.py:309: UserWarning: nn.init.xavier_normal is now deprecated in favor of nn.init.xavier_normal_.
  nn.init.xavier_normal(factor)
/content/MultiBench/fusions/common_fusions.py:316: UserWarning: nn.init.xavier_normal is now deprecated in favor of nn.init.xavier_normal_.
  nn.init.xavier_normal(self.fusion_weights)


Epoch 0 train loss: tensor(1.3221, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 0 valid loss: 1.386063814163208
Saving Best
Epoch 1 train loss: tensor(1.3203, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 1 valid loss: 1.3849766254425049
Saving Best
Epoch 2 train loss: tensor(1.3256, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 2 valid loss: 1.3866013288497925
Epoch 3 train loss: tensor(1.3193, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 3 valid loss: 1.3926748037338257
Epoch 4 train loss: tensor(1.3199, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 4 valid loss: 1.3800575733184814
Saving Best
Epoch 5 train loss: tensor(1.3250, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 5 valid loss: 1.3900824785232544
Epoch 6 train loss: tensor(1.3131, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 6 valid loss: 1.345107078552246
Saving Best
Epoch 7 train loss: tensor(1.1123, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 7 valid loss: 1.2136056423187256
Saving Best
Epoch 8 train loss: te

Answer the following:
1. (2 points) Sometimes when training you may notice the model gets stuck in a range of loss and never seems to get it's loss down. What does this suggest? What are some ways you can fix this?
2. (2 points) What are some other fusion methods we could use that we could use? Would they lead to improvements compared to early fusion?
3. (6 points) Explain the difference between early fusion techniques and late fusion techniques. Be sure to discuss their benefits and tradeoffs.

# Problem 5: Other Fusion Techniques (30 points)

Now, we want you to try implementing some of these fusion techniques on your dataset! For this part, you will implement these fusion techniques:
1. Early fusion
2. Late fusion
3. TensorFusion
4. Low-Rank Tensor (LMF) Fusion


**You cannot just import and use the functions available in Multibench to do this. In addition, use einsum where applicable. TO RECIEVE FULL CREDIT, THE FUSIONS YOU IMPLEMENTATION MUST WORK WITH THE DATASET YOU CREATED FROM HOMEWORK 1. YOU WILL HAVE TO CREATE A SIMPLE MODEL FOR EACH FUSION TECHNIQUE TO PLAY WITH.**

**(5 points)** In your write up, report the best validation accuracies of your multimodal model (don't forget to include what hyperparameters you included) after training and any modifications that had to be done to your data or model to be able to train on it. In addition, talk about which technique you believe would be best for your dataset and why that is.

**Design the fusion classes with the modalities you are specifically working with in mind. The example we worked through above with MOSI was meant as a showcase of fusion in action - we do not require you to use text, video and audio as the modalities. Use whichever ones you are working with!**

The code below provided is to be filled in with the models you set up for each technique. For an example, the first fusion technique has been done for you.

**We use audio (mel spectrogram) and vision (video frames) from the HD-EPIC egocentric kitchen dataset to predict verb classes. Our example video P01-20240202-110250 has 224 narration segments with 40 unique verb classes.**

In [ ]:
# === HD-EPIC Dataset Setup ===
# Imports
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.autograd import Variable
import torchvision.transforms as T
import numpy as np
import pandas as pd
import cv2
import os
import time
import subprocess
import math
from tqdm import tqdm

# ---- Configuration ----
VIDEO_PATH = "../hdepic_example/P01-20240202-110250.mp4"
NARRATIONS_PATH = "../hdepics_annotations/narrations-and-action-segments/HD_EPIC_Narrations.pkl"
VERB_CLASSES_PATH = "../hdepics_annotations/narrations-and-action-segments/HD_EPIC_verb_classes.csv"
EXAMPLE_VIDEO_ID = "P01-20240202-110250"

NUM_VERB_CLASSES = 106
IMG_SIZE = 112
N_MELS = 64
AUDIO_SR = 16000   # downsample for speed
EMBED_DIM = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# === HD-EPIC Dataset (single center frame + mel spectrogram) ===

# --- Audio extraction via ffmpeg (no torchaudio needed) ---
print("Extracting audio with ffmpeg...")
_proc = subprocess.run(
    ["ffmpeg", "-i", VIDEO_PATH, "-f", "s16le", "-ac", "1", "-ar", str(AUDIO_SR), "-"],
    capture_output=True, timeout=120)
_audio_raw = np.frombuffer(_proc.stdout, dtype=np.int16).astype(np.float32) / 32768.0
AUDIO_WAVEFORM = torch.from_numpy(_audio_raw).unsqueeze(0)  # (1, samples)
print(f"Audio: {AUDIO_WAVEFORM.shape[1]/AUDIO_SR:.1f}s at {AUDIO_SR}Hz")

# --- Precompute mel filterbank ---
_n_fft, _hop = 1024, 512
_n_freqs = _n_fft // 2 + 1
_mel_lo = 2595 * math.log10(1 + 0/700)
_mel_hi = 2595 * math.log10(1 + AUDIO_SR/2/700)
_mel_pts = torch.linspace(_mel_lo, _mel_hi, N_MELS + 2)
_hz_pts = 700 * (10**(_mel_pts/2595) - 1)
_bins = ((_n_fft+1) * _hz_pts / AUDIO_SR).long()
MEL_FBANK = torch.zeros(N_MELS, _n_freqs)
for _i in range(N_MELS):
    _lo, _mid, _hi = _bins[_i].item(), _bins[_i+1].item(), _bins[_i+2].item()
    for _j in range(_lo, _mid):
        MEL_FBANK[_i, _j] = (_j - _lo) / max(_mid - _lo, 1)
    for _j in range(_mid, _hi):
        MEL_FBANK[_i, _j] = (_hi - _j) / max(_hi - _mid, 1)
MEL_WINDOW = torch.hann_window(_n_fft)

def compute_mel(waveform):
    """Log-mel spectrogram using torch STFT + einsum mel filterbank."""
    stft = torch.stft(waveform, _n_fft, hop_length=_hop, window=MEL_WINDOW, return_complex=True)
    power = stft.abs().pow(2)
    mel = torch.einsum('mf,bft->bmt', MEL_FBANK, power)  # (1, N_MELS, T)
    mel = torch.log(mel + 1e-9)
    return F.interpolate(mel.unsqueeze(0), size=(N_MELS, N_MELS),
                        mode='bilinear', align_corners=False).squeeze(0)


# --- Pre-extract all data (single center frame + mel per segment) ---
print("\nPre-extracting all segments (single center frame + mel)...")
t0 = time.time()

narrations = pd.read_pickle(NARRATIONS_PATH)
segments = narrations[narrations['video_id'] == EXAMPLE_VIDEO_ID].reset_index(drop=True)
print(f"Segments: {len(segments)}")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_transform = T.Compose([
    T.ToPILImage(), T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

all_frames_list = []
all_mels_list = []
all_labels_list = []

for idx in range(len(segments)):
    row = segments.iloc[idx]
    start_sec, end_sec = row['start_timestamp'], row['end_timestamp']

    # Single center frame
    center_sec = (start_sec + end_sec) / 2
    center_frame = min(int(center_sec * fps), total_frames - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, center_frame)
    ret, frame = cap.read()
    if ret:
        frame_t = frame_transform(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    else:
        frame_t = torch.zeros(3, IMG_SIZE, IMG_SIZE)
    all_frames_list.append(frame_t)

    # Audio mel
    ss = int(start_sec * AUDIO_SR)
    es = min(int(end_sec * AUDIO_SR), AUDIO_WAVEFORM.shape[1])
    if es <= ss: es = ss + AUDIO_SR
    seg = AUDIO_WAVEFORM[:, ss:es]
    if seg.shape[1] < AUDIO_SR // 2:
        seg = F.pad(seg, (0, AUDIO_SR//2 - seg.shape[1]))
    all_mels_list.append(compute_mel(seg))

    # Multi-label verb target
    label = torch.zeros(NUM_VERB_CLASSES)
    for vc in row['verb_classes']:
        label[vc] = 1.0
    all_labels_list.append(label)

cap.release()

ALL_FRAMES = torch.stack(all_frames_list)   # (224, 3, 112, 112)
ALL_MELS = torch.stack(all_mels_list)       # (224, 1, 64, 64)
ALL_LABELS = torch.stack(all_labels_list)   # (224, 106)
print(f"Done in {time.time()-t0:.0f}s")
print(f"Frames: {ALL_FRAMES.shape}, Mels: {ALL_MELS.shape}, Labels: {ALL_LABELS.shape}")

# --- Cached dataset + split ---
class CachedDataset(Dataset):
    def __init__(self, frames, mels, labels):
        self.frames, self.mels, self.labels = frames, mels, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.frames[i], self.mels[i], self.labels[i]

n_total = len(ALL_LABELS)
n_train, n_val = int(0.7 * n_total), int(0.15 * n_total)
n_test = n_total - n_train - n_val
torch.manual_seed(42)
perm = torch.randperm(n_total)

train_set = CachedDataset(ALL_FRAMES[perm[:n_train]], ALL_MELS[perm[:n_train]], ALL_LABELS[perm[:n_train]])
val_set = CachedDataset(ALL_FRAMES[perm[n_train:n_train+n_val]], ALL_MELS[perm[n_train:n_train+n_val]], ALL_LABELS[perm[n_train:n_train+n_val]])
test_set = CachedDataset(ALL_FRAMES[perm[n_train+n_val:]], ALL_MELS[perm[n_train+n_val:]], ALL_LABELS[perm[n_train+n_val:]])
print(f"Split: train={n_train}, val={n_val}, test={n_test}")

BATCH_SIZE = 16
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

print(f"\nSample: frame={ALL_FRAMES[0].shape}, mel={ALL_MELS[0].shape}, "
      f"label={ALL_LABELS[0].shape} ({ALL_LABELS[0].sum():.0f} verbs)")


In [ ]:
# === Encoders for HD-EPIC (single frame) ===

class HDEPICVideoEncoder(nn.Module):
    """CNN encoder for a single video frame."""
    def __init__(self, output_dim=64):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),  # 112->56
            nn.MaxPool2d(2),                                                            # 56->28
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),  # 28->14
            nn.AdaptiveAvgPool2d(4),                                                    # 14->4
            nn.Flatten(),                                                               # 64*4*4=1024
        )
        self.fc = nn.Linear(1024, output_dim)

    def forward(self, x):
        # x: (batch, 3, H, W) — single frame
        return self.fc(self.cnn(x))


class HDEPICAudioEncoder(nn.Module):
    """CNN encoder for mel spectrograms."""
    def __init__(self, output_dim=64):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),  # 64->32
            nn.MaxPool2d(2),                                                            # 32->16
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),  # 16->8
            nn.AdaptiveAvgPool2d(4),                                                    # 8->4
            nn.Flatten(),                                                               # 64*4*4=1024
        )
        self.fc = nn.Linear(1024, output_dim)

    def forward(self, x):
        # x: (batch, 1, n_mels, n_mels)
        return self.fc(self.cnn(x))


# Verify
_v = HDEPICVideoEncoder(EMBED_DIM)
_a = HDEPICAudioEncoder(EMBED_DIM)
print(f"Video encoder: {_v(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape}")  # (2, 64)
print(f"Audio encoder: {_a(torch.randn(2, 1, N_MELS, N_MELS)).shape}")      # (2, 64)
del _v, _a


# Early Fusion

In [ ]:
class EarlyFusion(nn.Module):
    """Outer product fusion: combines modalities before any decision."""
    def __init__(self):
        super(EarlyFusion, self).__init__()

    def forward(self, x):
        # x = (video_emb, audio_emb), each (B, D)
        # Outer product using einsum -> (B, D, D)
        return torch.einsum('bi,bj->bij', x[0], x[1])

# (5 Points) Late Fusion

In [ ]:
class LateFusion(nn.Module):
    """Concatenation-based late fusion: each modality is processed independently,
    then their embeddings are concatenated for a joint decision."""
    def __init__(self):
        super(LateFusion, self).__init__()

    def forward(self, x):
        # x = (video_emb, audio_emb), each (B, D)
        # Simple concatenation along feature dim using einsum for reshaping
        # We use cat here as einsum doesn't naturally express concat;
        # einsum is used in the model's head instead
        return torch.cat([x[0], x[1]], dim=1)  # (B, 2D)

NameError: name 'nn' is not defined

# (5 points) Tensor Fusion

In [ ]:
class TensorFusion(nn.Module):
    """Full tensor fusion: computes the outer product of augmented (with bias) modality
    vectors, capturing all pairwise interactions including bias terms."""
    def __init__(self):
        super(TensorFusion, self).__init__()

    def forward(self, x):
        # x = (video_emb, audio_emb), each (B, D)
        # Augment with bias term (append 1)
        v_aug = torch.cat([x[0], torch.ones(x[0].size(0), 1, device=x[0].device)], dim=1)  # (B, D+1)
        a_aug = torch.cat([x[1], torch.ones(x[1].size(0), 1, device=x[1].device)], dim=1)  # (B, D+1)
        # Full outer product using einsum
        fused = torch.einsum('bi,bj->bij', v_aug, a_aug)  # (B, D+1, D+1)
        # Flatten the tensor product
        return fused.view(fused.size(0), -1)  # (B, (D+1)^2)

# (5 Points) Low-Rank Tensor Fusion (LMF) Fusion

In [ ]:
class LMFFusion(nn.Module):
    """Low-Rank Multimodal Fusion: approximates tensor fusion with low-rank factors,
    drastically reducing parameters. Uses einsum for efficient computation."""
    def __init__(self, dim_v, dim_a, output_dim, rank=4):
        super(LMFFusion, self).__init__()
        self.rank = rank
        self.output_dim = output_dim

        # Low-rank factors for each modality (+ bias)
        # W_v: (rank, D_v+1, output_dim) - video factor
        # W_a: (rank, D_a+1, output_dim) - audio factor
        self.W_v = nn.Parameter(torch.randn(rank, dim_v + 1, output_dim) * 0.01)
        self.W_a = nn.Parameter(torch.randn(rank, dim_a + 1, output_dim) * 0.01)
        self.bias = nn.Parameter(torch.zeros(output_dim))

    def forward(self, x):
        # x = (video_emb, audio_emb), each (B, D)
        B = x[0].size(0)

        # Augment with bias
        v_aug = torch.cat([x[0], torch.ones(B, 1, device=x[0].device)], dim=1)  # (B, D_v+1)
        a_aug = torch.cat([x[1], torch.ones(B, 1, device=x[1].device)], dim=1)  # (B, D_a+1)

        # Project each modality through its low-rank factor using einsum
        # h_v: (B, rank, output_dim) = einsum('bi,rio->bro', v_aug, W_v)
        h_v = torch.einsum('bi,rio->bro', v_aug, self.W_v)  # (B, rank, output_dim)
        h_a = torch.einsum('bi,rio->bro', a_aug, self.W_a)  # (B, rank, output_dim)

        # Element-wise product across modalities, then sum over rank
        # This approximates the full tensor product
        fused = torch.einsum('bro,bro->bo', h_v, h_a) + self.bias  # (B, output_dim)
        return fused

In [ ]:
# === Multimodal Fusion Models + Training/Evaluation ===

class MultimodalFusionModel(nn.Module):
    """Wraps video encoder + audio encoder + fusion + classification head."""
    def __init__(self, fusion_module, fusion_out_dim, num_classes=NUM_VERB_CLASSES,
                 embed_dim=EMBED_DIM, hidden_dim=128):
        super().__init__()
        self.video_encoder = HDEPICVideoEncoder(embed_dim)
        self.audio_encoder = HDEPICAudioEncoder(embed_dim)
        self.fusion = fusion_module
        self.head = nn.Sequential(
            nn.Linear(fusion_out_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, frame, audio):
        v_emb = self.video_encoder(frame)    # (B, D)
        a_emb = self.audio_encoder(audio)    # (B, D)
        fused = self.fusion((v_emb, a_emb))  # depends on fusion type
        return self.head(fused)


class UnimodalVideoModel(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, num_classes=NUM_VERB_CLASSES):
        super().__init__()
        self.encoder = HDEPICVideoEncoder(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes))
    def forward(self, frame, audio):
        return self.head(self.encoder(frame))


class UnimodalAudioModel(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, num_classes=NUM_VERB_CLASSES):
        super().__init__()
        self.encoder = HDEPICAudioEncoder(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes))
    def forward(self, frame, audio):
        return self.head(self.encoder(audio))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def topk_recall(logits, labels, k):
    """For each sample: how many true verbs appear in top-k predictions?"""
    _, topk = logits.topk(k, dim=1)
    hits, total = 0, 0
    for i in range(len(labels)):
        true_verbs = labels[i].nonzero(as_tuple=True)[0].tolist()
        pred_verbs = topk[i].tolist()
        hits += len(set(true_verbs) & set(pred_verbs))
        total += len(true_verbs)
    return hits / max(total, 1)


def train_fusion_model(name, model, num_epochs=30, lr=1e-3):
    """Train a fusion model and return metrics for comparison."""
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    n_params = count_parameters(model)
    print(f"\n{'='*60}")
    print(f"Training {name} | Params: {n_params:,}")
    print(f"{'='*60}")

    best_val_top3 = 0
    best_state = None
    train_losses = []
    start_time = time.time()
    convergence_epoch = num_epochs

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for frames, mel, labels in train_loader:
            frames, mel, labels = frames.to(device), mel.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(frames, mel), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)

        # Validate every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0:
            model.eval()
            all_logits, all_labels = [], []
            with torch.no_grad():
                for f, m, l in val_loader:
                    all_logits.append(model(f.to(device), m.to(device)).cpu())
                    all_labels.append(l)
            logits = torch.cat(all_logits)
            labels = torch.cat(all_labels)
            r1 = topk_recall(logits, labels, 1)
            r3 = topk_recall(logits, labels, 3)
            r5 = topk_recall(logits, labels, 5)
            print(f"  Ep {epoch+1:3d} | Loss: {avg_loss:.4f} | "
                  f"Top1: {r1:.3f} Top3: {r3:.3f} Top5: {r5:.3f}")

            if r3 > best_val_top3:
                best_val_top3 = r3
                convergence_epoch = epoch
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

    elapsed = time.time() - start_time

    # Test with best model
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for f, m, l in test_loader:
            all_logits.append(model(f.to(device), m.to(device)).cpu())
            all_labels.append(l)
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    t1 = topk_recall(logits, labels, 1)
    t3 = topk_recall(logits, labels, 3)
    t5 = topk_recall(logits, labels, 5)
    print(f"  --- Test Top1: {t1:.3f} Top3: {t3:.3f} Top5: {t5:.3f}")

    mem_mb = sum(p.nelement() * p.element_size() for p in model.parameters()) / 1024**2

    return {
        'name': name, 'params': n_params,
        'val_top3': best_val_top3,
        'test_top1': t1, 'test_top3': t3, 'test_top5': t5,
        'convergence_epoch': convergence_epoch,
        'time_sec': elapsed, 'mem_mb': mem_mb,
        'train_losses': train_losses,
    }


# === Run all experiments ===
D = EMBED_DIM
results = []

# Unimodal baselines
results.append(train_fusion_model('Video Only', UnimodalVideoModel()))
results.append(train_fusion_model('Audio Only', UnimodalAudioModel()))

# Early Fusion: outer product -> (B, D*D)
results.append(train_fusion_model('Early Fusion',
    MultimodalFusionModel(EarlyFusion(), D * D)))

# Late Fusion: concatenation -> (B, 2D)
results.append(train_fusion_model('Late Fusion',
    MultimodalFusionModel(LateFusion(), 2 * D)))

# Tensor Fusion: augmented outer product -> (B, (D+1)^2)
results.append(train_fusion_model('Tensor Fusion',
    MultimodalFusionModel(TensorFusion(), (D + 1) ** 2)))

# LMF Fusion: low-rank -> (B, lmf_out)
lmf_out = 128
results.append(train_fusion_model('LMF Fusion',
    MultimodalFusionModel(LMFFusion(D, D, lmf_out, rank=4), lmf_out)))

# Summary table
print(f"\n{'='*90}")
print(f"{'Model':<18} {'Params':>10} {'Val Top3':>10} {'Test Top1':>10} "
      f"{'Test Top3':>10} {'Test Top5':>10} {'Time(s)':>8} {'Mem(MB)':>8}")
print(f"{'-'*90}")
for r in results:
    print(f"{r['name']:<18} {r['params']:>10,} {r['val_top3']:>10.3f} {r['test_top1']:>10.3f} "
          f"{r['test_top3']:>10.3f} {r['test_top5']:>10.3f} {r['time_sec']:>8.0f} {r['mem_mb']:>8.3f}")


In [ ]:
# === Visualizations: Parameters, Memory, Convergence, Accuracy ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
names = [r['name'] for r in results]
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#00BCD4']

# 1. Number of parameters
axes[0, 0].barh(names, [r['params'] for r in results], color=colors[:len(results)])
axes[0, 0].set_xlabel('Number of Parameters')
axes[0, 0].set_title('Model Parameters')
axes[0, 0].ticklabel_format(axis='x', style='scientific', scilimits=(0,0))

# 2. Memory use
axes[0, 1].barh(names, [r['mem_mb'] for r in results], color=colors[:len(results)])
axes[0, 1].set_xlabel('Memory (MB)')
axes[0, 1].set_title('Model Memory Use')

# 3. Training loss curves (convergence)
for i, r in enumerate(results):
    axes[1, 0].plot(r['train_losses'], label=r['name'], color=colors[i])
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Training Loss')
axes[1, 0].set_title('Convergence (Training Loss)')
axes[1, 0].legend(fontsize=7)

# 4. Test accuracy comparison (Top-1, Top-3, Top-5)
x = np.arange(len(names))
w = 0.25
axes[1, 1].bar(x - w, [r['test_top1'] for r in results], w, label='Top-1', color='#2196F3')
axes[1, 1].bar(x,     [r['test_top3'] for r in results], w, label='Top-3', color='#4CAF50')
axes[1, 1].bar(x + w, [r['test_top5'] for r in results], w, label='Top-5', color='#FF9800')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_title('Test Verb Recall (Top-k)')
axes[1, 1].legend()
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('part5_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Convergence epoch comparison
print("\nConvergence (epoch of best val Top-3):")
for r in results:
    print(f"  {r['name']:<18} epoch {r['convergence_epoch']+1}")

print("\nPros and cons:")
print("- Unimodal: fewer params, faster, but miss cross-modal info")
print("- Early/Tensor fusion: capture fine interactions but high param count")
print("- Late fusion: simple, low params, often competitive")
print("- LMF: good balance — captures interactions with controlled rank")


(10 points) In addition, create some visualizations of the following for each fusion:

* Number of parameters for each model (unimodal and multimodal)
* Memory Use
* Time until convergence

You are free to plot them here or through other means (like wandb). After doing so, discuss what are the pros and cons of unimodal versus multimodal models.

# Problem 6: Contrastive Learning (30 points)

For the next part of this HW, we will focus on contrastive learning. As a reminder, contrastive learning is a local, discrete alignment method used in machine learning. To explore this, we look at [CLIP](https://arxiv.org/pdf/2103.00020), a multimodal model developed by OpenAI that uses contrastive learning to align visual and textual data together.

**THIS IS JUST AN EXAMPLE, DO NOT LET THIS RESTRICT THE IMPLEMENTATION YOU WILL BE DOING.**

In [ ]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-8a206fek
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-8a206fek
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=ca8a5f278ed012078baeed3a2a84a3db5796719ed214ab98c5fae7bc4d192588
  Stored in directory: /tmp/pip-ephem-wheel-cache-ju96rty9/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [ ]:
# Packages to import
import transformers
import torch
import clip
from PIL import Image
import requests
from io import BytesIO

First, we create the model.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP on {device}...")
model, preprocess = clip.load("ViT-B/32", device=device)

Loading CLIP on cuda...


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 161MiB/s]


Next, we will load an image to use. Note that we cannot use the MOSI dataset - we need to use raw data and the data points from the dataset already have extracted features. Upload a picture of someone smiling to use for this example (you can just find one online, save it and add to here).

In [ ]:
image_filename = "../smiling_person.png" # REPLACE WITH YOUR FILE
image = Image.open(image_filename).convert("RGB")

Now, we will prepare the prompt to use.

In [ ]:
# Options to pick from
text_options = ["a photo of a sad person", "a photo of a happy person", "a photo of an angry person"]
image_input = preprocess(image).unsqueeze(0).to(device)
text_inputs = clip.tokenize(text_options).to(device)

Now, let's run the inference and get the results!

In [ ]:
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_inputs)

    # Normalize features
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Calculate similarity (Dot Product)
    similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
    values, indices = similarity[0].topk(3)

print(f"\nImage classified against: {text_options}")
print("-" * 30)
for value, index in zip(values, indices):
    print(f"{text_options[index]:>30s}: {100 * value.item():.2f}%")


Image classified against: ['a photo of a sad person', 'a photo of a happy person', 'a photo of an angry person']
------------------------------
     a photo of a happy person: 98.44%
    a photo of an angry person: 1.09%
       a photo of a sad person: 0.49%


(10 pts) We will now create, train and run zero-shot classification using contrastive learning for your own dataset. Fill in the missing information below for a generalize contrastive learning model. The training and zero-shot classification functions have been provided to you, through you may need to make slight modifications based on your dataset setup. **Design the model keeping in mind the modalities that you are specifically using. THE CLIP EXAMPLE ABOVE IS JUST TO SHOW CONTRASTIVE LEARNING IN ACTION - WE ARE NOT REQUIRING THAT YOU USE TEXT AND IMAGE AS THE MODALITIES OF CHOICE.** Try various queries, projectors, and settings on your dataset!

**You must use einsum where applicable.**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

# === Dataset: expand multi-label segments into (frame, verb_id) pairs ===
class ImageVerbPairDataset(Dataset):
    """Uses pre-cached ALL_FRAMES and segment verb_classes to create pairs."""
    def __init__(self, frames, segments_df, indices):
        self.pairs = []  # (frame_tensor, verb_id)
        for i in indices:
            row = segments_df.iloc[i]
            for vc in row['verb_classes']:
                self.pairs.append((frames[i], vc))
        print(f"  ImageVerbPairDataset: {len(self.pairs)} pairs from {len(indices)} segments")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        frame, verb_id = self.pairs[idx]
        return frame, verb_id


# Build pair datasets using the same split indices from Part 5
n_total = len(ALL_LABELS)
n_train, n_val = int(0.7 * n_total), int(0.15 * n_total)
n_test = n_total - n_train - n_val
torch.manual_seed(42)
_perm = torch.randperm(n_total).tolist()

train_indices = _perm[:n_train]
val_indices = _perm[n_train:n_train+n_val]
test_indices = _perm[n_train+n_val:]

cl_train_ds = ImageVerbPairDataset(ALL_FRAMES, segments, train_indices)
cl_val_ds = ImageVerbPairDataset(ALL_FRAMES, segments, val_indices)
cl_test_ds = ImageVerbPairDataset(ALL_FRAMES, segments, test_indices)

cl_train_loader = DataLoader(cl_train_ds, batch_size=32, shuffle=True, num_workers=0)
cl_val_loader = DataLoader(cl_val_ds, batch_size=32, shuffle=False, num_workers=0)
cl_test_loader = DataLoader(cl_test_ds, batch_size=32, shuffle=False, num_workers=0)


# === Contrastive model: Image <-> Verb alignment ===
class ImageVerbCLModel(nn.Module):
    def __init__(self, num_verbs=NUM_VERB_CLASSES, embed_dim=EMBED_DIM,
                 projected_dim=32, temp=0.07):
        super(ImageVerbCLModel, self).__init__()
        # 1. Image encoder (single center frame)
        self.image_encoder = HDEPICVideoEncoder(embed_dim)

        # 1b. Verb encoder: learnable embedding table for each verb class
        self.verb_embedding = nn.Embedding(num_verbs, embed_dim)

        # 2. Projectors into shared contrastive space
        self.image_projector = nn.Sequential(
            nn.Linear(embed_dim, projected_dim),
            nn.ReLU(),
            nn.Linear(projected_dim, projected_dim),
        )
        self.verb_projector = nn.Sequential(
            nn.Linear(embed_dim, projected_dim),
            nn.ReLU(),
            nn.Linear(projected_dim, projected_dim),
        )

        # 3. Learnable temperature
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / temp))

    def forward(self, frame, verb_ids):
        # 1. Extract features
        image_feat = self.image_encoder(frame)            # (B, embed_dim)
        verb_feat = self.verb_embedding(verb_ids)         # (B, embed_dim)

        # 2. Project to shared space
        image_emb = self.image_projector(image_feat)      # (B, projected_dim)
        verb_emb = self.verb_projector(verb_feat)         # (B, projected_dim)

        # 3. L2 normalize
        image_emb = F.normalize(image_emb, dim=1)
        verb_emb = F.normalize(verb_emb, dim=1)

        return image_emb, verb_emb


# Contrastive loss using einsum similarity matrix
class ContrastiveLoss(nn.Module):
    def __init__(self):
        super(ContrastiveLoss, self).__init__()
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, image_emb, verb_emb, logit_scale):
        B = image_emb.size(0)
        # Clamp scale for stability
        scale = logit_scale.exp().clamp(max=100.0)
        # Similarity matrix using einsum
        sim = torch.einsum('id,jd->ij', image_emb, verb_emb) * scale  # (B, B)
        labels = torch.arange(B, device=image_emb.device)
        # Symmetric loss
        loss = (self.loss_fn(sim, labels) + self.loss_fn(sim.T, labels)) / 2
        return loss


print("ImageVerbCLModel and ContrastiveLoss defined.")
print(f"Verb embedding covers {NUM_VERB_CLASSES} verb classes")

In [ ]:
import torch.optim as optim

def train_contrastive(model, contrastive_loss, train_loader, val_loader,
                       num_epochs=50, lr=5e-4, device='cpu'):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    model.to(device)
    print(f"Starting image-verb contrastive training for {num_epochs} epochs...")

    best_val_loss = float('inf')
    best_state = None
    for epoch in range(num_epochs):
        # Train
        model.train()
        epoch_loss = 0.0
        for frame, verb_ids in train_loader:
            frame, verb_ids = frame.to(device), verb_ids.to(device)
            optimizer.zero_grad()
            image_emb, verb_emb = model(frame, verb_ids)
            loss = contrastive_loss(image_emb, verb_emb, model.logit_scale)
            loss.backward()
            optimizer.step()
            # Clamp temperature for stability
            with torch.no_grad():
                model.logit_scale.clamp_(max=np.log(100))
            epoch_loss += loss.item()

        avg_train = epoch_loss / max(len(train_loader), 1)

        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for frame, verb_ids in val_loader:
                frame, verb_ids = frame.to(device), verb_ids.to(device)
                image_emb, verb_emb = model(frame, verb_ids)
                val_loss += contrastive_loss(image_emb, verb_emb, model.logit_scale).item()
        avg_val = val_loss / max(len(val_loader), 1)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d} | Train Loss: {avg_train:.4f} | "
                  f"Val Loss: {avg_val:.4f} | Scale: {model.logit_scale.exp().item():.2f}")

    model.load_state_dict(best_state)
    return model


# Train
cl_model = ImageVerbCLModel(num_verbs=NUM_VERB_CLASSES, embed_dim=EMBED_DIM,
                             projected_dim=32, temp=0.07)
cl_loss = ContrastiveLoss()
cl_model = train_contrastive(cl_model, cl_loss, cl_train_loader, cl_val_loader,
                              num_epochs=50, lr=5e-4, device=str(device))

In [ ]:
# === Zero-shot verb prediction: given image, find closest verb ===
@torch.no_grad()
def zero_shot_verb_from_image(model, test_loader, device, top_k=3):
    """For each image, compute similarity to ALL verb class embeddings
    and predict the top-k verbs. No training labels needed at inference."""
    model.eval()

    # Get all verb class embeddings
    all_verb_ids = torch.arange(NUM_VERB_CLASSES, device=device)
    verb_feat = model.verb_embedding(all_verb_ids)              # (106, embed_dim)
    verb_emb = model.verb_projector(verb_feat)                  # (106, projected_dim)
    verb_emb = F.normalize(verb_emb, dim=1)                    # (106, projected_dim)

    correct = 0
    total = 0
    all_preds = []
    all_trues = []

    for frame, verb_ids in test_loader:
        frame = frame.to(device)
        image_feat = model.image_encoder(frame)
        image_emb = model.image_projector(image_feat)
        image_emb = F.normalize(image_emb, dim=1)  # (B, projected_dim)

        # Similarity of each image to all 106 verb classes using einsum
        sim = torch.einsum('bd,cd->bc', image_emb, verb_emb)  # (B, 106)

        # Top-k predicted verb classes
        _, top_preds = sim.topk(top_k, dim=1)  # (B, top_k)

        for i in range(len(verb_ids)):
            true_verb = verb_ids[i].item()
            preds = top_preds[i].cpu().tolist()
            all_preds.append(preds)
            all_trues.append(true_verb)
            if true_verb in preds:
                correct += 1
            total += 1

    acc = correct / max(total, 1)
    print(f"  Top-{top_k} accuracy: {acc:.4f} ({correct}/{total})")
    return acc, all_preds, all_trues


# Load verb names
verb_df = pd.read_csv(VERB_CLASSES_PATH)
verb_map = dict(zip(verb_df['id'], verb_df['key']))

print("=== Zero-Shot Verb Prediction from Image ===")
for k in [1, 3, 5]:
    zero_shot_verb_from_image(cl_model, cl_test_loader, device, top_k=k)


# === Visualization: image-verb embedding space ===
print("\n=== Embedding Space Visualization ===")
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

cl_model.eval()

# Get all verb class embeddings
with torch.no_grad():
    all_verb_ids = torch.arange(NUM_VERB_CLASSES, device=device)
    verb_feat = cl_model.verb_embedding(all_verb_ids)
    verb_emb = cl_model.verb_projector(verb_feat)
    verb_emb = F.normalize(verb_emb, dim=1).cpu().numpy()  # (106, D)

# Get test image embeddings
test_image_embs, test_verb_ids = [], []
with torch.no_grad():
    for frame, vids in cl_test_loader:
        frame = frame.to(device)
        i_feat = cl_model.image_encoder(frame)
        i_emb = cl_model.image_projector(i_feat)
        i_emb = F.normalize(i_emb, dim=1)
        test_image_embs.append(i_emb.cpu().numpy())
        test_verb_ids.append(vids.numpy())

test_images = np.concatenate(test_image_embs)  # (N_test, D)
test_vids = np.concatenate(test_verb_ids)       # (N_test,)

# PCA on combined verb + image embeddings
active_verbs = sorted(set(test_vids.tolist()))
active_verb_emb = verb_emb[active_verbs]  # (n_active, D)

combined = np.concatenate([active_verb_emb, test_images], axis=0)
pca = PCA(n_components=2)
proj = pca.fit_transform(combined)
n_verbs = len(active_verbs)
proj_verbs = proj[:n_verbs]
proj_images = proj[n_verbs:]

# Map test verb ids to color indices
verb_to_color = {v: i for i, v in enumerate(active_verbs)}
image_colors = [verb_to_color[v] for v in test_vids]

fig, ax = plt.subplots(1, 1, figsize=(12, 9))

# Plot image embeddings (small dots)
scatter = ax.scatter(proj_images[:, 0], proj_images[:, 1], c=image_colors,
                     cmap='tab20', marker='o', s=30, alpha=0.5, label='Image segments')

# Plot verb class embeddings (large stars with labels)
verb_colors = list(range(n_verbs))
ax.scatter(proj_verbs[:, 0], proj_verbs[:, 1], c=verb_colors,
           cmap='tab20', marker='*', s=200, edgecolors='black', linewidths=1,
           label='Verb classes', zorder=5)

# Label each verb star
for i, v in enumerate(active_verbs):
    ax.annotate(verb_map.get(v, str(v)), (proj_verbs[i, 0], proj_verbs[i, 1]),
                fontsize=7, fontweight='bold', ha='center', va='bottom',
                xytext=(0, 6), textcoords='offset points')

# Draw lines from image to its true verb
for i in range(len(test_vids)):
    v_idx = verb_to_color[test_vids[i]]
    ax.plot([proj_images[i, 0], proj_verbs[v_idx, 0]],
            [proj_images[i, 1], proj_verbs[v_idx, 1]],
            'k-', alpha=0.05, linewidth=0.5)

ax.legend(fontsize=10)
ax.set_title('Contrastive Alignment: Image Segments (o) vs Verb Classes (*)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.savefig('part6_embedding_space.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: part6_embedding_space.png")


# === Similarity heatmap: verb predictions per test image ===
fig2, ax2 = plt.subplots(1, 1, figsize=(14, 8))
with torch.no_grad():
    all_verb_ids_t = torch.arange(NUM_VERB_CLASSES, device=device)
    v_emb_all = F.normalize(cl_model.verb_projector(cl_model.verb_embedding(all_verb_ids_t)), dim=1)

    # Collect all test similarities
    all_sims, all_true = [], []
    for frame, vids in cl_test_loader:
        frame = frame.to(device)
        i_emb = F.normalize(cl_model.image_projector(cl_model.image_encoder(frame)), dim=1)
        sim = torch.einsum('bd,cd->bc', i_emb, v_emb_all)
        all_sims.append(sim.cpu())
        all_true.extend(vids.tolist())

sim_matrix = torch.cat(all_sims).numpy()  # (N_test, 106)

# Show only active verb columns for readability
active_names = [verb_map.get(v, str(v)) for v in active_verbs]
sim_active = sim_matrix[:, active_verbs]  # (N_test, n_active)

# Sort rows by true verb for visual grouping
sort_idx = np.argsort(all_true)
sim_sorted = sim_active[sort_idx]

im = ax2.imshow(sim_sorted, aspect='auto', cmap='viridis', interpolation='nearest')
ax2.set_xticks(range(len(active_names)))
ax2.set_xticklabels(active_names, rotation=60, ha='right', fontsize=7)
ax2.set_ylabel('Test samples (sorted by true verb)')
ax2.set_xlabel('Verb classes')
ax2.set_title('Image-Verb Similarity Heatmap (test set)')
plt.colorbar(im, ax=ax2, label='Cosine similarity')
plt.tight_layout()
plt.savefig('part6_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: part6_similarity_heatmap.png")


# === Alignment examples ===
print("\n=== Alignment Examples ===")
with torch.no_grad():
    all_verb_ids_t = torch.arange(NUM_VERB_CLASSES, device=device)
    v_emb_all = F.normalize(cl_model.verb_projector(cl_model.verb_embedding(all_verb_ids_t)), dim=1)

    for frame, vids in cl_test_loader:
        frame = frame.to(device)
        i_emb = F.normalize(cl_model.image_projector(cl_model.image_encoder(frame)), dim=1)
        sim = torch.einsum('bd,cd->bc', i_emb, v_emb_all)
        _, top3 = sim.topk(3, dim=1)

        print("\nSample predictions (true verb -> top-3 predicted):")
        for i in range(min(15, len(vids))):
            true = verb_map.get(vids[i].item(), '?')
            preds = [verb_map.get(p.item(), '?') for p in top3[i]]
            match = 'OK' if vids[i].item() in top3[i].cpu().tolist() else 'X '
            print(f"  {match} true='{true}' -> predicted={preds}")
        break

Now answer some of these questions:

1. (5 points) Any suprising results from using this on your dataset?
2. (5 points) Typically, cross-entropy loss is used in this contrastive learning, why is this the case?
3. (10 points) Create some visual examples of the data post alignment. Can you point out samples where the alignment worked and where it failed? Why do you suspect that is?

# Problem 7: Reflection (10 points)

Now we'll take some time to reflect on this homework. Take some time to discuss the following:

1. (5 points) What concept did you find the most interesting?
2. (5 points) Which concepts (if any) do you see being useful towards your goal? Why? If there was none, discuss why.
3. (0 points, optional) Is there a topic that was discussed during lectures up to the release of the assignment that you wished was covered in the homework? Any from the assignment that you wanted there to be touched upon more? Any feedback you have in general for homeworks or the class?